# PathSnap — Notebook 07: HMM + Viterbi route reconstruction (Milestone 7)
**Owner: C (nisarg-ravi)** · uses the selected, Bayesian-tuned classifier from M6 (`hist_gb_tuned`).

**Design (`pathsnap/hmm.py`)**
| Element | Our choice | Why |
|---|---|---|
| Hidden state at GPS point t | one of the candidate arcs of t (same rule as training: d_nearest + 20 m, cap 150 m) | keeps candidate ↔ arc correspondence; ~5 states per point |
| Emission | log P(arc correct \| features) from hist_gb | the classifier's job; selected partly for calibration (ECE 0.005) |
| Transition | −\|d_route − d_gps\| / β_t, β_t = β·max(1, Δt)^γ | Newson & Krumm (2009): driving distance between consecutive matches ≈ straight-line GPS distance. d_route uses the **directed** graph (one-way streets, U-turns cost a detour) |
| Same arc | forward distance along the arc; backward ≤ 15 m allowed (jitter while crawling) | a parked car's jitter must not force arc changes |
| Impossible transition | target not reachable within 3·d_gps + 200 m → probability 0 | route-continuity constraint |
| GPS gaps | Δt > break_s, or nothing reachable → cut the chain, decode the next segment independently | trips 78/90 have 15 h / 42 h gaps; route breaks (11 trips) & disconnected network parts (M2) |
| No candidates | point gets no prediction (−1), chain is cut | rare (0 of labelled points) |
| Decoding | Viterbi in log space per segment | exact most-likely path |

**Decoder parameters (β, break_s, γ)** are tuned on the **80 training trips** using the classifier's
**out-of-fold** probabilities (from M6's grouped CV), so the decoder never sees probabilities from a model trained on
the same trip. Student-test is used once for the final classifier-only vs classifier + HMM comparison.

In [ ]:
import os, sys, json, time, zipfile, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    ZIP_PATH, EXTRACT_DIR = f"{DRIVE}/Student folder.zip", "/content/pathsnap_data"
    DS_DIR, M4, M6, OUT = f"{DRIVE}/m3_outputs/dataset", f"{DRIVE}/m4_outputs", f"{DRIVE}/m6_outputs", f"{DRIVE}/m7_outputs"
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO): os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO if os.path.isdir(f"{REPO}/pathsnap") else DRIVE
else:
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip"); EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    DS_DIR = os.environ.get("PATHSNAP_DS", "m3_outputs/dataset"); M6 = os.environ.get("PATHSNAP_M6", "m6_outputs")
    M4 = os.environ.get("PATHSNAP_M4", "m4_outputs")
    OUT = os.environ.get("PATHSNAP_M7", "m7_outputs"); PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
DATA_ROOT = next(dp for dp, dn, _ in os.walk(EXTRACT_DIR) if "train" in dn and "student_test" in dn)
from pathsnap.modeling import load_dataset
from pathsnap.io import list_trips, trip_dir
from pathsnap.hmm import TripLattice, viterbi, reconstruct_route
os.makedirs(OUT, exist_ok=True)
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
F, P = load_dataset(DS_DIR)
# emission-model candidates from M6 (all hist_gb; see notebook 06 §6)
CANDIDATES = {"hist_gb": M4,            # untuned (M4)
              "hist_gb_tuned": M6,      # best CV Top-1 in the Bayesian search
              "hist_gb_tuned_ll": M6}   # lowest log loss among Top-1-tied trials

In [ ]:
# ---- 1. Lattices (candidates, emissions, route-distance costs) + helpers -----------------------------------------
def build(part, probs, col):
    out = {}
    for tid in list_trips(DATA_ROOT, part):
        o = probs[probs.trip_id == tid]
        out[tid] = TripLattice(trip_dir(DATA_ROOT, part, tid), dict(zip(zip(o.pt.values, o.arc.values), o[col].values)))
    return out

def evaluate(lats, part, pred_fn, full=True):
    """pred_fn(lat) -> predicted arc per point. Returns pooled metrics + per-trip table.
    full=False: Top-1 and strict continuity only (fast; used inside the tuning grid)."""
    rows, tot = [], {"lab": 0, "ok": 0, "pairs": 0, "adj": 0, "reach": 0}
    for tid, lat in lats.items():
        pred = pred_fn(lat)
        pts = P[(P.partition == part) & (P.trip_id == tid)].sort_values("pt")
        lab = pts.label_status.values == 0; ref = pts.ref_arc.values
        ok = int((pred[lab] == ref[lab]).sum())
        A = lat.net.arcs; v = (pred[:-1] >= 0) & (pred[1:] >= 0) & (lat.dt[:-1] <= 60)
        a, b = pred[:-1][v], pred[1:][v]
        adj = (a == b) | (A[a, 1] == A[b, 0])
        idx = np.where(v)[0] if full else np.array([], int)
        reach = np.array([np.isfinite(lat.cost[i][np.where(lat.cand_arc[i] == pa)[0][0], np.where(lat.cand_arc[i + 1] == pb)[0][0]])
                          if lat.cost[i] is not None else False for i, pa, pb in zip(idx, a, b)], bool)
        ref_set = set(ref[lab]); pred_set = set(pred[lab & (pred >= 0)])
        tp = len(ref_set & pred_set); pr = tp / max(len(pred_set), 1); rc = tp / max(len(ref_set), 1)
        rows.append({"trip_id": tid, "labelled_points": int(lab.sum()), "top1": ok / max(lab.sum(), 1),
                     "continuity_strict": adj.mean() if len(adj) else np.nan, "continuity_reachable": reach.mean() if len(reach) else np.nan,
                     "route_arc_precision": pr, "route_arc_recall": rc, "route_arc_f1": 2 * pr * rc / max(pr + rc, 1e-9)})
        tot["lab"] += int(lab.sum()); tot["ok"] += ok; tot["pairs"] += len(adj); tot["adj"] += int(adj.sum()); tot["reach"] += int(reach.sum())
    T = pd.DataFrame(rows)
    if not full:
        return {"top1": tot["ok"] / tot["lab"], "continuity_strict": tot["adj"] / tot["pairs"]}, T
    return {"top1": tot["ok"] / tot["lab"], "continuity_strict": tot["adj"] / tot["pairs"], "continuity_reachable": tot["reach"] / tot["pairs"],
            "route_arc_f1_mean_trip": T.route_arc_f1.mean(), "trip_top1_mean": T.top1.mean()}, T

clf_only = lambda lat: np.array([a[np.argmax(lp)] if len(a) else -1 for a, lp in zip(lat.cand_arc, lat.logp)])

## 1b. Which tuned configuration should feed the HMM? (decided on TRAINING trips, out-of-fold)
The Bayesian search (M6) left three hist_gb configurations within noise of each other on classifier Top-1. We decode
the 80 training trips with each one's out-of-fold probabilities (small decoder grid) and keep the one with the best
Top-1 **after** HMM decoding, i.e. the system-level objective. Ties (< 0.001) → lower log loss.

In [ ]:
SMALL = [(b, 300.0, 0.0) for b in (0.5, 1, 2, 5)]
choice_rows = []
for name, d in CANDIDATES.items():
    oof_c = pd.read_csv(f"{d}/{name}_oof.csv.gz")
    lat_c = build("train", oof_c, "oof_proba")
    clf_c, _ = evaluate(lat_c, "train", clf_only, full=False)
    best_c = max(((evaluate(lat_c, "train", lambda lat: viterbi(lat, beta=b, break_s=k, gap_exp=g)[0], full=False)[0]["top1"], b) for b, k, g in SMALL))
    r = json.load(open(f"{d}/{name}_results.json"))
    choice_rows.append({"emission_model": name, "oof_top1_classifier": clf_c["top1"], "oof_top1_after_hmm": best_c[0], "best_beta": best_c[1],
                        "cv_log_loss": r["cv"]["val_log_loss"]["mean"], "infer_ms_per_1000_pts": r["inference_ms_per_1000_points"]})
    del lat_c
CH = pd.DataFrame(choice_rows).sort_values("oof_top1_after_hmm", ascending=False); CH.to_csv(f"{OUT}/00_emission_model_choice.csv", index=False)
display(CH.round(4))
top = CH[CH.oof_top1_after_hmm >= CH.oof_top1_after_hmm.max() - 0.001]
MODEL = top.sort_values("cv_log_loss").emission_model.iloc[0]
print("emission model for the HMM:", MODEL)
M_DIR = CANDIDATES[MODEL]
OOF = pd.read_csv(f"{M_DIR}/{MODEL}_oof.csv.gz"); TEP = pd.read_csv(f"{M_DIR}/{MODEL}_student_test_proba.csv.gz")
t0 = time.time(); LAT_TR = build("train", OOF, "oof_proba"); t1 = time.time(); LAT_TE = build("student_test", TEP, "proba")
print(f"lattices for {MODEL}: train {t1-t0:.0f}s, student-test {time.time()-t1:.0f}s")

## 2. Tune the decoder on the 80 training trips (out-of-fold probabilities)
Grid: β (transition scale, metres) × break_s (chain cut on gaps) × γ (gap scaling). Objective: pooled Top-1 on
labelled training points (same primary metric as the classifier).

In [ ]:
GRID = {"beta": [0.25, 0.5, 1, 2, 3, 5, 10, 20, 40, 80], "break_s": [30, 60, 120, 300], "gap_exp": [0.0, 0.5, 1.0]}   # β grid extended below 2 after a first run put the optimum on the edge
grid_rows = []
t0 = time.time()
for beta, brk, ge in itertools.product(*GRID.values()):
    m, _ = evaluate(LAT_TR, "train", lambda lat: viterbi(lat, beta=beta, break_s=brk, gap_exp=ge)[0], full=False)
    grid_rows.append({"beta": beta, "break_s": brk, "gap_exp": ge, **m})
G = pd.DataFrame(grid_rows).sort_values("top1", ascending=False); G.to_csv(f"{OUT}/01_decoder_grid_train.csv", index=False)
print(f"{len(G)} decoder settings evaluated in {time.time()-t0:.0f}s"); display(G.head(10).round(4))
BEST = G.iloc[0][["beta", "break_s", "gap_exp"]].to_dict(); BEST = {"beta": float(BEST["beta"]), "break_s": float(BEST["break_s"]), "gap_exp": float(BEST["gap_exp"])}
print("best decoder parameters (train):", BEST)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for brk in GRID["break_s"]:
    g = G[(G.break_s == brk) & (G.gap_exp == BEST["gap_exp"])].sort_values("beta"); ax[0].plot(g.beta, g.top1, "o-", label=f"break {brk}s")
base_tr, _ = evaluate(LAT_TR, "train", clf_only); ax[0].axhline(base_tr["top1"], ls="--", c="k", label="classifier only")
ax[0].set(xscale="log", xlabel="β (m)", ylabel="train Top-1 (OOF)", title=f"Decoder tuning (γ = {BEST['gap_exp']})"); ax[0].legend()
ax[1].scatter(G.continuity_strict, G.top1, c=np.log(G.beta), cmap="viridis"); ax[1].set(xlabel="strict continuity (train)", ylabel="Top-1", title="All settings (colour = log β)")
plt.tight_layout(); plt.savefig(f"{OUT}/01_decoder_tuning.png", dpi=130); plt.show()
print("train classifier-only:", {k: round(v, 4) for k, v in base_tr.items()})

## 3. Classifier-only vs classifier + HMM (student-test, once, with the tuned decoder)

In [ ]:
res = {}
res["classifier only"], T_clf = evaluate(LAT_TE, "student_test", clf_only)
t0 = time.time()
res["classifier + HMM"], T_hmm = evaluate(LAT_TE, "student_test", lambda lat: viterbi(lat, **BEST)[0])
dec_s = time.time() - t0
CMP = pd.DataFrame(res).T; CMP.loc["change"] = CMP.loc["classifier + HMM"] - CMP.loc["classifier only"]
CMP.to_csv(f"{OUT}/02_clf_vs_hmm_student_test.csv"); display(CMP.round(4))
PT = T_clf[["trip_id", "labelled_points", "top1", "continuity_strict", "continuity_reachable", "route_arc_f1"]].merge(
     T_hmm[["trip_id", "top1", "continuity_strict", "continuity_reachable", "route_arc_f1"]], on="trip_id", suffixes=("_clf", "_hmm"))
PT["top1_gain"] = PT.top1_hmm - PT.top1_clf; PT.to_csv(f"{OUT}/03_per_trip_student_test.csv", index=False); display(PT.round(4))
print(f"HMM decoding time, all student-test trips: {dec_s:.2f}s (lattice building excluded)")
# same comparison on train (OOF) for a 90-trip picture
res_tr_hmm, T_tr_hmm = evaluate(LAT_TR, "train", lambda lat: viterbi(lat, **BEST)[0])
_, T_tr_clf = evaluate(LAT_TR, "train", clf_only)
TT = T_tr_clf[["trip_id", "top1"]].merge(T_tr_hmm[["trip_id", "top1"]], on="trip_id", suffixes=("_clf", "_hmm")); TT["gain"] = TT.top1_hmm - TT.top1_clf
TT.to_csv(f"{OUT}/04_per_trip_train_oof.csv", index=False)
print("train (OOF): classifier-only %.4f → HMM %.4f | trips improved %d, unchanged %d, worse %d" % (
      base_tr["top1"], res_tr_hmm["top1"], (TT.gain > 1e-9).sum(), (TT.gain.abs() <= 1e-9).sum(), (TT.gain < -1e-9).sum()))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
k = ["top1", "continuity_strict", "continuity_reachable", "route_arc_f1_mean_trip"]; x = np.arange(len(k))
ax[0].bar(x - .2, CMP.loc["classifier only", k], .4, label="classifier only"); ax[0].bar(x + .2, CMP.loc["classifier + HMM", k], .4, label="classifier + HMM")
ax[0].set_xticks(x); ax[0].set_xticklabels(k, rotation=15); ax[0].set(ylim=(0.8, 1.0), title="Student-test (10 trips)"); ax[0].legend()
ax[1].scatter(TT.top1_clf, TT.top1_hmm, s=18, label="train trips (OOF)"); ax[1].scatter(PT.top1_clf, PT.top1_hmm, s=40, c="r", label="student-test trips")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(xlabel="Top-1 classifier only", ylabel="Top-1 with HMM", title="Per trip: above the diagonal = HMM helped"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/02_clf_vs_hmm.png", dpi=130); plt.show()

## 4. Where did the HMM help, and where not?

In [ ]:
ctx = F[F.partition == "student_test"].drop_duplicates(["trip_id", "pt"])[["trip_id", "pt", "heading_known"]]
rows = []
for tid, lat in LAT_TE.items():
    pc = clf_only(lat); ph = viterbi(lat, **BEST)[0]
    pts = P[(P.partition == "student_test") & (P.trip_id == tid)].sort_values("pt").merge(ctx, on=["trip_id", "pt"])
    A = lat.net.arcs
    for i, r in enumerate(pts.itertuples()):
        if r.label_status != 0: continue
        ref = r.ref_arc
        def kind(p):
            if p == ref: return "correct"
            if p >= 0 and tuple(A[p]) == tuple(A[ref][::-1]): return "twin"
            return "other wrong"
        rows.append({"trip_id": tid, "moving": bool(r.heading_known), "clf": kind(pc[i]), "hmm": kind(ph[i])})
E = pd.DataFrame(rows)
summary = pd.DataFrame({
    "classifier only": [(E.clf == "correct")[E.moving].mean(), (E.clf == "correct")[~E.moving].mean(), (E.clf == "twin").mean(), (E.clf == "other wrong").mean()],
    "classifier + HMM": [(E.hmm == "correct")[E.moving].mean(), (E.hmm == "correct")[~E.moving].mean(), (E.hmm == "twin").mean(), (E.hmm == "other wrong").mean()]},
    index=["Top-1 when moving", "Top-1 when stopped/slow", "share twin errors", "share other errors"])
summary.to_csv(f"{OUT}/05_where_hmm_helps.csv"); display(summary.round(4))
trans = pd.crosstab(E.clf, E.hmm); trans.to_csv(f"{OUT}/06_clf_to_hmm_transitions.csv"); display(trans)
print("points fixed by HMM:", int(((E.clf != "correct") & (E.hmm == "correct")).sum()), "| points broken by HMM:", int(((E.clf == "correct") & (E.hmm != "correct")).sum()))
print("trips where the HMM did not help (student-test):", PT[PT.top1_gain <= 0][["trip_id", "top1_clf", "top1_hmm"]].round(4).values.tolist())

## 4b. Self-checks before trusting the numbers
1. **No label leakage:** lattices are built with `read_trip(..., with_route=False)`; the reference route is only read
   by `evaluate`. Training-trip probabilities are out-of-fold; student-test probabilities come from a model fitted on
   training trips only. The decoder was tuned on training trips only.
2. **Every prediction is a real candidate** of its point.
3. **Segments / breaks**: how often the chain is cut, and why.
4. **The training trips where the HMM made things worse.**

In [ ]:
for tid, lat in list(LAT_TE.items()):
    ph, sg = viterbi(lat, **BEST)
    assert all(p == -1 or p in set(c) for p, c in zip(ph, lat.cand_arc)), "prediction outside candidate set"
print("check 2 passed: every HMM prediction is one of its point's candidates (student-test)")
seg_rows = []
for part, lats in (("train", LAT_TR), ("student_test", LAT_TE)):
    for tid, lat in lats.items():
        ph, sg = viterbi(lat, **BEST)
        gaps = int((lat.dt[:-1] > BEST["break_s"]).sum())
        seg_rows.append({"partition": part, "trip_id": tid, "points": len(ph), "segments": int(sg.max() + 1),
                         "cuts_by_gap": gaps, "cuts_unreachable": int(sg.max()) - gaps, "points_without_prediction": int((ph < 0).sum())})
SEG = pd.DataFrame(seg_rows); SEG.to_csv(f"{OUT}/07_segments.csv", index=False)
display(SEG.groupby("partition")[["segments", "cuts_by_gap", "cuts_unreachable", "points_without_prediction"]].agg(["sum", "max"]))
print("trips with the most segments:", SEG.sort_values("segments", ascending=False).head(5)[["partition", "trip_id", "segments", "cuts_by_gap", "cuts_unreachable"]].values.tolist())
W = TT[TT.gain < 0].merge(P[(P.partition == "train")].groupby("trip_id").agg(points=("pt", "size"), labelled=("label_status", lambda s: (s == 0).mean())), on="trip_id")
W = W.merge(SEG[SEG.partition == "train"][["trip_id", "segments", "cuts_unreachable"]], on="trip_id")
display(W.round(4))
print("→ inspect these with route_map(trip, 'train') below; typical causes are listed in the M7 summary.")

## 5. Folium maps: GPS, network, reference route, classifier-only and HMM routes

In [ ]:
import folium
def route_map(tid, part="student_test"):
    lat = LAT_TE[tid] if part == "student_test" else LAT_TR[tid]
    ph = viterbi(lat, **BEST)[0]; pc = clf_only(lat); full = reconstruct_route(lat, ph)
    pts = P[(P.partition == part) & (P.trip_id == tid)].sort_values("pt")
    nd, A = lat.nodes_lonlat, lat.net.arcs
    seg = lambda a: [[nd[A[a, 0], 1], nd[A[a, 0], 0]], [nd[A[a, 1], 1], nd[A[a, 1], 0]]]
    m = folium.Map(location=[lat.lonlat[:, 1].mean(), lat.lonlat[:, 0].mean()], zoom_start=14, tiles="cartodbpositron", control_scale=True)
    g_ref = folium.FeatureGroup(name="Reference route (green)")
    for a in sorted(set(pts[pts.label_status == 0].ref_arc)):
        folium.PolyLine(seg(a), color="green", weight=8, opacity=.45).add_to(g_ref)
    g_hmm = folium.FeatureGroup(name="Predicted route: classifier + HMM (blue)")
    for a in full: folium.PolyLine(seg(a), color="#1f4fd8", weight=3, opacity=.9).add_to(g_hmm)
    g_clf = folium.FeatureGroup(name="Classifier-only wrong picks (orange)", show=False)
    for i, a in enumerate(pc):
        if a >= 0 and pts.label_status.values[i] == 0 and a != pts.ref_arc.values[i]:
            folium.PolyLine(seg(a), color="orange", weight=3, opacity=.8).add_to(g_clf)
    g_gps = folium.FeatureGroup(name="Raw GPS points (grey)")
    for i in np.linspace(0, len(lat.lonlat) - 1, min(2500, len(lat.lonlat))).astype(int):
        folium.CircleMarker([lat.lonlat[i, 1], lat.lonlat[i, 0]], radius=2, color="#555555", fill=True, fill_opacity=.7, weight=0).add_to(g_gps)
    for g in (g_ref, g_hmm, g_clf, g_gps): g.add_to(m)
    m.fit_bounds([[lat.lonlat[:, 1].min(), lat.lonlat[:, 0].min()], [lat.lonlat[:, 1].max(), lat.lonlat[:, 0].max()]])
    folium.LayerControl(collapsed=False).add_to(m)
    r = PT.set_index("trip_id").loc[tid]
    m.get_root().html.add_child(folium.Element(f"""<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;
      padding:8px 10px;border:1px solid #999;border-radius:6px;font-size:13px"><b>{part} trip {tid}</b><br>
      <span style="color:green">&#9473;&#9473;</span> reference route<br><span style="color:#1f4fd8">&#9473;&#9473;</span> HMM route
      (Top-1 {r.top1_hmm:.3f})<br><span style="color:orange">&#9473;&#9473;</span> classifier-only wrong picks (Top-1 {r.top1_clf:.3f})<br>
      <span style="color:#555">&#9679;</span> raw GPS</div>"""))
    return m
for tid in PT.sort_values("top1_gain", ascending=False).trip_id.values[:3].tolist() + [int(PT.sort_values("top1_gain").trip_id.values[0])]:
    route_map(int(tid)).save(f"{OUT}/route_trip_{int(tid):08d}.html"); print("saved map", tid)

## 6. Save the decoder configuration (used by the API in M9)

In [ ]:
cfg = {"emission_model": MODEL, "decoder": BEST, "back_tol_m": 15.0, "reach_limit": "3*d_gps + 200 m",
       "max_dt_for_transitions_s": 600, "tuned_on": "80 training trips, out-of-fold probabilities",
       "train_oof_top1": {"classifier_only": base_tr["top1"], "hmm": res_tr_hmm["top1"]},
       "student_test": CMP.drop(index="change").to_dict(orient="index")}
json.dump(cfg, open(f"{OUT}/hmm_config.json", "w"), indent=1, default=float); print(json.dumps(cfg, indent=1, default=float)[:900])